# Projeto 2 — Assistente de IA para Segurança Digital para Adolescentes 🛡️
**Fica Esperto AI: chatbot educativo de prevenção a golpes para adolescentes**

**Objetivo:** protótipo funcional que orienta adolescentes sobre golpes, links falsos, senhas, Pix, Wi-Fi público, privacidade e notícias falsas, usando uma base de conhecimento revisada, com linguagem e exemplos próximos da realidade desse público, e que encaminha questões desconhecidas.

- **Público-alvo:** adolescentes e jovens que usam redes sociais, aplicativos de mensagens e plataformas digitais no dia a dia.
- **Questão de pesquisa:** um assistente virtual pode auxiliar adolescentes a reconhecer golpes e adotar práticas mais seguras no ambiente digital?
- **Hipótese 1:** o assistente será capaz de responder corretamente dúvidas frequentes sobre segurança digital de adolescentes.
- **Hipótese 2:** informações em linguagem simples e objetiva ajudam adolescentes a compreender melhor os riscos digitais e as formas de prevenção.
- **Baseline:** o chatbot do notebook de exemplo da professora. **Melhoria:** adaptar o chatbot ao público adolescente.

> **Importante:** esta é uma demonstração acadêmica, **sem integração com canais oficiais** de bancos, escolas ou empresas. **Não compartilhe senhas, códigos, CPF, documentos, telefones ou dados bancários neste protótipo.**

*Este notebook segue a mesma estrutura do notebook de exemplo (`Projeto_2_Seguranca_Digital_Exemplo_Funcional.ipynb`).*

## 0. Abra e execute no Colab
Acesse [colab.research.google.com](https://colab.research.google.com) → **Arquivo → Fazer upload de notebook** → escolha este `.ipynb` → **Conectar**. Antes de executar, abra o painel **Arquivos** (ícone de pasta) e faça o upload de `Projeto_2_Seguranca_Digital_Adolescentes.csv`. Depois execute na ordem com **Shift + Enter**.

**Conceitos:** *chatbot* é uma interface de conversa; *intenção* (ou tema) é o assunto provável do usuário; *TF-IDF* transforma textos em números; *similaridade de cosseno* mede quão próximos dois textos são; *classificador* é um modelo que aprende, a partir de exemplos, a qual tema um texto pertence (aqui usamos *regressão logística*); *fallback* é a resposta prudente quando não há informação segura.

## 1. Entender a base que acompanha o notebook

| Coluna | Para que serve |
|---|---|
| `id_caso` | Identificador do caso, no formato `5_<tema>_<número>` |
| `pergunta` | Dúvida do usuário (sintética, não é log de conversa real) |
| `tema` | Categoria esperada (8 temas) |
| `resposta_revisada` | Resposta educativa revisada do tema |
| `publico`, `canal` | Público e canal a que a base se destina |
| `conjunto` | `treino` (para o modelo aprender) ou `teste` (para avaliar) |
| `criticidade` | `alta` quando envolve dinheiro ou acesso à conta, `moderada` nos demais casos |

Além da base da disciplina, o projeto tem dois arquivos criados pelo grupo (frases **sintéticas**, como no exemplo):

| CSV | Para que serve |
|---|---|
| `exemplos_complementares.csv` | 210 frases de treino escritas do jeito que adolescentes escrevem: 164 sobre os 8 temas e 46 sobre outros assuntos (categoria `fora_do_escopo`), no mesmo formato do `exemplos_intencoes.csv` do exemplo |
| `perguntas_validacao.csv` | 104 perguntas novas para avaliar o assistente, em três lotes; **nunca são usadas no treino** |

Faça o upload dos três CSVs no Colab.

In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline, make_union

pd.set_option("display.max_colwidth", 120)

CAMINHO_CSV = "Projeto_2_Seguranca_Digital_Adolescentes.csv"
df_bruto = pd.read_csv(CAMINHO_CSV)

exemplos = pd.read_csv("exemplos_complementares.csv")
validacao = pd.read_csv("perguntas_validacao.csv")

for titulo, tabela in [("Base da disciplina", df_bruto), ("Exemplos complementares", exemplos),
                       ("Perguntas de validação", validacao)]:
    print(titulo, "—", len(tabela), "linhas")
    display(tabela.head(3))
print("\nDivisão treino/teste da base:")
print(df_bruto["conjunto"].value_counts())

## 2. Validar qualidade e segurança da base
Não treinaremos o modelo com dados privados: as perguntas são sintéticas. Antes de treinar, verificamos se cada caso tem tema e resposta revisada.

In [ ]:
print("Valores ausentes por coluna:")
print(df_bruto.isna().sum(), end="\n\n")
print("Valores distintos de 'tema' (repr mostra espaços extras):")
print(df_bruto["tema"].map(repr).value_counts())

**Problemas encontrados:** 1 linha sem `tema` (`5_boato_09`) e variações de escrita do mesmo tema (`"WIFI"` e `"  wifi  "`). Como o `id_caso` traz o tema, usamos ele para corrigir.

In [ ]:
df = df_bruto.copy()
df["tema"] = df["tema"].str.strip().str.lower()
tema_do_id = df["id_caso"].str.extract(r"^\d+_(.+)_\d+$")[0]
df["tema"] = df["tema"].fillna(tema_do_id)

assert (df["tema"] == tema_do_id).all(), "tema diferente do indicado no id_caso"
assert df[["pergunta", "tema", "resposta_revisada"]].notna().all().all()
assert df.groupby("tema")["resposta_revisada"].nunique().eq(1).all(), "cada tema deve ter uma única resposta revisada"
assert set(df["conjunto"]) == {"treino", "teste"}
assert df["tema"].nunique() >= 8

treino = df[df["conjunto"] == "treino"].reset_index(drop=True)
teste = df[df["conjunto"] == "teste"].reset_index(drop=True)

situacao = lambda p: re.sub(r"^(O que devo fazer se|Como agir quando|Que cuidados tomar se)\s+", "", p)
repetidas = set(teste["pergunta"].map(situacao)) & set(treino["pergunta"].map(situacao))

# arquivos do grupo: mesmos temas da base e nenhuma pergunta de avaliação dentro do treino
assert exemplos["exemplo_id"].is_unique
CATEGORIAS = set(df["tema"]) | {"fora_do_escopo"}  # os 8 temas + a categoria de perguntas sobre outros assuntos
assert exemplos["intencao"].isin(CATEGORIAS).all() and validacao["tema_esperado"].isin(CATEGORIAS).all()
textos_treino = set(exemplos["texto_usuario"].str.lower())
assert not textos_treino & set(validacao["pergunta"].str.lower()), "pergunta de validação no treino"
assert not textos_treino & set(teste["pergunta"].str.lower()), "pergunta de teste no treino"

print("✅ Bases validadas.")
print(df["tema"].value_counts().to_string())
print(f"\ntreino: {len(treino)} | teste: {len(teste)} | situações do teste repetidas no treino: {len(repetidas)}")

**Base organizada.** No arquivo original as linhas estão embaralhadas. Esta célula salva uma cópia limpa e organizada em `base_limpa.csv`, ordenada por tema e pelo número do caso, sem alterar o CSV original. O resto do notebook usa a base já limpa (`df`).

In [ ]:
df["numero"] = df["id_caso"].str.extract(r"_(\d+)$")[0].astype(int)
base_limpa = df.sort_values(["tema", "numero"]).drop(columns="numero").reset_index(drop=True)
df = df.drop(columns="numero")
base_limpa.to_csv("base_limpa.csv", index=False, encoding="utf-8-sig")

print("Salvo: base_limpa.csv —", len(base_limpa), "linhas, ordenadas por tema e número do caso\n")
print(pd.crosstab(base_limpa["tema"], base_limpa["conjunto"], margins=True, margins_name="total"))
display(base_limpa.head(5))

Cada tema tem **uma única resposta revisada**, então acertar o tema = entregar a resposta correta. As perguntas de teste descrevem **situações que não aparecem no treino**: o teste mede se o chatbot generaliza, e não se decorou frases.

## 3. Construir o mecanismo de inteligência artificial
Assim como no exemplo, **não enviamos mensagens a um modelo generativo externo**. O chatbot **recupera respostas predefinidas e revisadas**, o que reduz o risco de inventar procedimentos ou prometer estornos. É um protótipo de **IA de recuperação textual**, não um grande modelo de linguagem (LLM).

Construímos dois mecanismos para comparar:
- **3.1 Baseline:** o mecanismo do notebook de exemplo, aplicado à nossa base.
- **3.2 Assistente adaptado (melhoria):** classificador de temas, gírias e respostas para adolescentes.

**Fluxo:** pergunta → detectar dados sensíveis → normalizar texto e gírias → verificar se o assunto está na base → classificar o tema → consultar conteúdo revisado → responder ou encaminhar.

### 3.1 Baseline: mecanismo do notebook de exemplo
Mesmo código do exemplo: TF-IDF + similaridade de cosseno, palavras-chave por tema (`PALAVRAS_TEMA`), bloqueio de dados sensíveis e *fallback* com os mesmos limiares. A única mudança é a base: as frases de treino são as perguntas de **treino** do nosso CSV, e as palavras-chave são as do exemplo para os 8 temas que existem aqui (o tema `boato` não existia no exemplo).

In [ ]:
vetorizador = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2))
matriz = vetorizador.fit_transform(treino["pergunta"])
resposta_por_tema = df.drop_duplicates("tema").set_index("tema")["resposta_revisada"]

# Bloqueio orientativo: apenas um aviso, sem armazenar o texto da conversa.
PADROES_DADOS_SENSIVEIS = [
    r"\b\d{3}\.\d{3}\.\d{3}-?\d{2}\b",  # CPF formatado
    r"\b\d{11}\b",  # possível CPF sem pontuação
    r"\b(?:senha|password|token|codigo de verificacao|código de verificação)\s*[:=]\s*\S+",
]


def contem_dado_sensivel(texto):
    return any(re.search(p, texto, flags=re.IGNORECASE) for p in PADROES_DADOS_SENSIVEIS)


PALAVRAS_TEMA = {
    "phishing": ["phishing", "link suspeito", "mensagem falsa"],
    "pix": ["pix", "transferi dinheiro", "transferência fraudulenta"],
    "senha": ["senha", "senhas", "password"],
    "whatsapp": ["whatsapp"],
    "privacidade": ["privacidade", "cpf", "dados pessoais"],
    "wifi": ["wifi", "wi-fi", "rede pública"],
    "falso_suporte": ["falso_suporte", "suporte remoto", "acesso remoto"],
}


def detectar_tema_explicito(pergunta):
    texto = pergunta.lower()
    hits = []
    for tema, expressoes in PALAVRAS_TEMA.items():
        for palavra in expressoes:
            if re.search(r"(?<!\w)" + re.escape(palavra) + r"(?!\w)", texto):
                hits.append((len(palavra), tema))
    return max(hits)[1] if hits else None


FALLBACK = ("Não tenho uma orientação suficientemente confiável para esse caso. "
            "Evite clicar em links ou fornecer códigos. Procure o canal oficial da instituição envolvida "
            "e, em situação de risco ou perda financeira, contate imediatamente seu banco.")


temas_treino = treino["tema"]


# vetorizador, matriz e temas_treino viram parâmetros para a comparação da seção 5.4
def baseline_responder(pergunta, limiar=0.22, vetorizador=vetorizador, matriz=matriz, temas_treino=temas_treino):
    if contem_dado_sensivel(pergunta):
        return {"tema": "dado_sensivel", "confianca": 0.0,
                "resposta": "⚠️ Parece que sua mensagem contém um dado sensível. Reformule sem informações pessoais."}
    tema_direto = detectar_tema_explicito(pergunta)
    similaridades = cosine_similarity(vetorizador.transform([pergunta]), matriz)[0]
    indice = int(np.argmax(similaridades))
    score = float(similaridades[indice])
    palavras_golpe = ["golpe", "fraude", "internet", "seguranca", "segurança", "invad"]
    if tema_direto is None and (score < max(limiar, 0.48) or
                                (score < 0.90 and not any(t in pergunta.lower() for t in palavras_golpe))):
        return {"tema": "fora_do_escopo", "confianca": score, "resposta": FALLBACK}
    tema = tema_direto or temas_treino.iloc[indice]
    return {"tema": tema, "confianca": score, "resposta": resposta_por_tema[tema]}


r = baseline_responder("Um amigo me mandou mensagem de outro número pedindo dinheiro, o que faço?")
print("Tema:", r["tema"], "| similaridade:", round(r["confianca"], 3))
print(r["resposta"])

### 3.2 Melhoria: assistente adaptado para adolescentes
Três mudanças em relação ao baseline:

1. **Entende melhor a pergunta:** normaliza o texto (minúsculas, sem acentos), remove o começo repetido das perguntas ("O que devo fazer se...") e troca gírias por termos da base ("zap" → "whatsapp", "hackearam" → "descoberto minha senha"). O tema é identificado por um **classificador** (regressão logística) treinado com as perguntas e com a base de conhecimento; ele usa pedaços de palavras (n-gramas de caracteres), o que tolera erros de digitação.
2. **Fala a língua do público:** cada tema tem uma resposta curta, em passos, com um exemplo do dia a dia do adolescente e a indicação de pedir ajuda a um adulto de confiança. O conteúdo segue a resposta revisada da base. Como no exemplo, a resposta termina com **Atenção** e **Referência para consulta**.
3. **Sabe quando não sabe:** se a pergunta não tem relação com a base, o assistente não chuta um tema; ele avisa a limitação e indica onde buscar ajuda.

In [ ]:
PREFIXOS = r"^(o que devo fazer se|como agir quando|que cuidados tomar se|o que fazer se|o que eu faco se|como agir se)\s+"

# gíria/variação -> termo presente na base
GIRIAS = {
    r"\bzap\b|\bzapzap\b|\bwpp\b|\bwhats\b": "whatsapp",
    r"\binsta\b|\binstagram\b|\btiktok\b|\bdiscord\b": "aplicativo",
    r"\bwifi\b|\bwi fi\b|\bwi-fi\b": "wi-fi",
    r"\bfake news\b|\bfake\b": "noticia falsa",
    r"\bprint\b|\bprints\b": "imagem",
    r"\bcodigo de 6 digitos\b|\bcodiguinho\b|\bcodigo (que|q) chegou\b": "codigo de verificacao",
    r"\bpix\b": "pix transferencia",
    r"\bclonaram\b|\bclonado\b|\bhackearam\b|\bhackeado\b|\binvadiram\b|\broubaram\b": "descoberto minha senha",
    r"\bmae\b|\bpai\b|\bvo\b|\bvovo\b|\btia\b|\btio\b|\bprimo\b|\bprima\b": "parente",
    r"\bmano\b|\bvei\b|\bmds\b|\bpfv\b|\bpf\b|\bkkk+\b": "",
}


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(PREFIXOS, "", texto)
    for padrao, troca in GIRIAS.items():
        texto = re.sub(padrao, troca, texto)
    texto = re.sub(r"[^a-z0-9\- ]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


print(normalizar("Mano, minha mãe mandou msg no zap pedindo Pix, o que faço?"))

As perguntas de treino da base cobrem só umas 4 situações por tema, cada uma escrita de 3 jeitos formais. Com isso, nos primeiros testes o assistente acertou apenas 55% de perguntas novas escritas em linguagem informal. Por isso o classificador aprende com três fontes:
1. as perguntas de **treino** da base da disciplina;
2. os **exemplos complementares**: 164 frases no jeito adolescente de escrever e 46 perguntas sobre outros assuntos (jogos, escola, música, comida), que ensinam o modelo a reconhecer o que está **fora do escopo**;
3. a **base de conhecimento**: a resposta revisada de cada tema e um pequeno vocabulário de termos do assunto.

In [ ]:
VOCABULARIO = {
    "whatsapp": "whatsapp mensagem contato amigo parente familiar se passou por perfil falso numero novo pediu dinheiro emergencia",
    "wifi": "wi-fi rede aberta rede publica internet gratuita conexao shopping aeroporto cafe escola",
    "pix": "pix transferencia banco dinheiro chave comprovante fraude vendedor pagamento devolucao",
    "boato": "noticia falsa boato fonte compartilhar grupo viral video imagem verificar",
    "senha": "senha codigo de verificacao duas etapas conta login compartilhar senha trocar senha",
    "phishing": "link suspeito sms e-mail site falso premio clicar dados bancarios atualizacao cadastral conta bloqueada",
    "falso_suporte": "suporte atendente central de atendimento ligacao acesso remoto instalar aplicativo pede dados funcionario da empresa",
    "privacidade": "privacidade permissoes camera microfone localizacao fotos perfil dados pessoais informacoes pessoais",
}

base_conhecimento = (
    df.drop_duplicates("tema")[["tema", "resposta_revisada"]]
    .assign(pergunta=lambda d: d["resposta_revisada"] + " " + d["tema"].map(VOCABULARIO))
)
complementares = exemplos.rename(columns={"texto_usuario": "pergunta", "intencao": "tema"})
treino_assistente = pd.concat([treino[["pergunta", "tema"]], complementares[["pergunta", "tema"]],
                               base_conhecimento[["pergunta", "tema"]]])

classificador = make_pipeline(
    make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
    ),
    LogisticRegression(C=10, max_iter=2000),
)
classificador.fit(treino_assistente["pergunta"].map(normalizar), treino_assistente["tema"])
print(len(treino_assistente), "textos de treino (perguntas da base + exemplos complementares + base de conhecimento)")

**Respostas adaptadas.** O texto principal de cada tema foi reescrito para adolescentes a partir da `resposta_revisada`. Os campos **Atenção** e **Referência para consulta** vêm da base de conhecimento do notebook de exemplo, para os temas que existem nos dois. O tema `boato` não existe no exemplo: o alerta foi escrito pelo grupo e a referência é a Cartilha de Segurança para Internet do CERT.br.

In [ ]:
RESPOSTAS_ADOLESCENTES = {
    "whatsapp": (
        "🚨 Cuidado: isso tem cara de golpe de perfil falso.\n"
        "1. Não mande dinheiro, Pix nem nenhum código que chegou por SMS.\n"
        "2. Ligue para a pessoa no número que você já tinha, ou fale com ela pessoalmente.\n"
        "3. Se for golpe, bloqueie e denuncie o contato no próprio app.\n"
        "Exemplo: \"Oi, troquei de número, me faz um Pix rapidinho?\" → ligue no número antigo antes de qualquer coisa.\n"
        "Conte para um adulto de confiança, principalmente se o golpista usou a foto de alguém da sua família."
    ),
    "wifi": (
        "📶 Wi-Fi público é prático, mas não é seguro para tudo.\n"
        "1. Não entre no app do banco nem faça compras numa rede aberta.\n"
        "2. Para coisas importantes, use o 4G/5G do seu celular.\n"
        "3. Pergunte no local qual é o nome certo da rede, porque golpistas criam redes com nomes parecidos.\n"
        "Exemplo: no shopping apareceu \"Shopping_Free\" e \"Shopping-Free-2\"? Confirme com o balcão antes de conectar.\n"
        "Se a rede pedir dados pessoais demais para liberar a internet, desconfie e não preencha."
    ),
    "pix": (
        "💸 Dinheiro envolvido: aja rápido.\n"
        "1. Se você já transferiu ou acha que é fraude, fale com seu banco AGORA, pelo app oficial ou pelo telefone que está no cartão.\n"
        "2. Guarde os comprovantes e tire prints da conversa.\n"
        "3. Pergunte ao banco sobre os procedimentos de contestação e devolução.\n"
        "Exemplo: um vendedor pediu para pagar por Pix fora do site da loja? Não pague: a proteção da plataforma deixa de valer.\n"
        "Avise um adulto responsável, porque ele pode ajudar a falar com o banco."
    ),
    "boato": (
        "🧐 Antes de repassar, confira.\n"
        "1. Não compartilhe nada que você ainda não verificou.\n"
        "2. Procure o assunto em sites de notícia confiáveis e em canais oficiais.\n"
        "3. Veja a data e quem publicou. Notícia velha ou sem fonte é sinal de alerta.\n"
        "Exemplo: \"COMPARTILHE ANTES QUE APAGUEM!!!\" → pressa e urgência são truques comuns de boato.\n"
        "Se for print ou vídeo viral, desconfie ainda mais: imagens podem ser editadas ou tiradas de contexto."
    ),
    "senha": (
        "🔐 Sua senha é só sua.\n"
        "1. Use uma senha longa (uma frase funciona bem) e diferente para cada conta.\n"
        "2. Ative a verificação em duas etapas nas redes sociais e no e-mail.\n"
        "3. Nunca passe senha ou código de verificação para ninguém, nem para amigos ou namorado(a).\n"
        "Exemplo: \"MeuGatoComeBolo3x!\" é bem mais forte que \"123456\" ou a sua data de nascimento.\n"
        "Acha que alguém descobriu sua senha? Troque na hora e encerre as sessões abertas em outros aparelhos."
    ),
    "phishing": (
        "🎣 Link suspeito é isca.\n"
        "1. Não clique e não digite sua senha em sites que chegaram por mensagem, SMS ou e-mail.\n"
        "2. Quer conferir? Abra o app oficial ou digite você mesmo o endereço do site.\n"
        "3. Denuncie a mensagem como spam ou golpe quando o app permitir.\n"
        "Exemplo: \"Você ganhou uma skin grátis, clique aqui!\" ou \"Sua conta será bloqueada\" → é golpe para roubar sua conta.\n"
        "Empresas sérias não pedem senha nem dados do banco por mensagem."
    ),
    "falso_suporte": (
        "🛑 Suporte de verdade não pede código nem acesso ao seu celular.\n"
        "1. Encerre a conversa ou desligue a ligação.\n"
        "2. Não passe códigos, não instale apps que pedirem e não permita acesso remoto.\n"
        "3. Se precisar de ajuda, procure o suporte pelos canais oficiais da empresa (app ou site oficial).\n"
        "Exemplo: \"Sou do suporte do jogo, me passa o código que chegou para eu recuperar sua conta\" → é golpe.\n"
        "Na dúvida, mostre a conversa para um adulto de confiança antes de fazer qualquer coisa."
    ),
    "privacidade": (
        "👀 Você decide quem vê o quê.\n"
        "1. Revise as permissões dos apps (câmera, microfone, localização, contatos) nas configurações do celular.\n"
        "2. Deixe o perfil privado e escolha quem pode ver suas fotos e stories.\n"
        "3. Compartilhe só o necessário e tire acessos que não fazem sentido.\n"
        "Exemplo: um app de lanterna pedindo acesso à câmera e aos contatos? Negue.\n"
        "Postou endereço, escola ou telefone sem querer? Apague o post e avise um adulto de confiança."
    ),
}

# Atenção e referência: base de conhecimento do notebook de exemplo (boato: escrito pelo grupo)
ALERTAS = {
    "whatsapp": "Não transfira dinheiro com base apenas em mensagem, áudio ou foto de perfil.",
    "wifi": "Não ignore alertas de segurança do navegador.",
    "pix": "Não transfira outro valor para liberar estorno.",
    "boato": "Não compartilhe conteúdo sem verificar a fonte, mesmo que tenha vindo de amigos.",
    "senha": "Nunca compartilhe senhas nem códigos de autenticação.",
    "phishing": "Não abra links suspeitos nem envie códigos.",
    "falso_suporte": "Nunca entregue senha ou código por telefone.",
    "privacidade": "Não publique CPF, endereço, localização em tempo real ou documentos.",
}
REFERENCIAS = {
    "whatsapp": "https://faq.whatsapp.com",
    "wifi": "https://cartilha.cert.br",
    "pix": "https://www.bcb.gov.br",
    "boato": "https://cartilha.cert.br",
    "senha": "https://cartilha.cert.br",
    "phishing": "https://www.gov.br/cert",
    "falso_suporte": "https://cartilha.cert.br",
    "privacidade": "https://www.gov.br/anpd",
}
NOMES_TEMAS = {
    "whatsapp": "Golpe no WhatsApp", "wifi": "Wi-Fi público", "pix": "Pix", "boato": "Notícia falsa",
    "senha": "Senhas", "phishing": "Link falso", "falso_suporte": "Falso suporte", "privacidade": "Privacidade",
}

RESPOSTA_FORA_DO_ESCOPO = (
    "🤔 Não tenho certeza de que entendi, ou esse assunto não está na minha base.\n"
    "Eu sei ajudar com: golpes no WhatsApp, Pix, links falsos, falso suporte, senhas, Wi-Fi público, "
    "privacidade e notícias falsas. Tente explicar de outro jeito.\n"
    "Se você está passando por uma situação difícil (ameaça, exposição de fotos, cyberbullying), "
    "converse com um adulto de confiança. Você também pode procurar o canal de ajuda da SaferNet Brasil."
)
RESPOSTA_DADO_SENSIVEL = (
    "⚠️ Opa! Parece que sua mensagem tem um dado pessoal (como CPF, senha ou código).\n"
    "Nunca mande isso em chat nenhum, nem aqui. Apague e escreva sua dúvida sem esses dados."
)

for dicionario in (RESPOSTAS_ADOLESCENTES, ALERTAS, REFERENCIAS, NOMES_TEMAS):
    assert set(dicionario) == set(df["tema"]), "falta conteúdo para algum tema"
print("✅ Conteúdo completo para os", len(RESPOSTAS_ADOLESCENTES), "temas.")

**Quando a pergunta está fora do escopo?** A confiança do classificador sozinha não funciona bem: com 8 temas, perguntas válidas e curtas também recebem confiança baixa. Por isso a regra combina dois sinais.

**Palavras características da base:** palavras que aparecem no treino de no máximo 2 temas. Não contam palavras genéricas ("quero", "meu", "fazer") nem palavras que aparecem em quase todos os temas ("jogo", "aplicativo", "celular").

A pergunta é tratada como fora do escopo quando:
1. a **categoria "fora do escopo"** é a mais provável **e** tem probabilidade de 0,4 ou mais, ou a pergunta não tem nenhuma palavra característica; **ou**
2. a pergunta **não tem nenhuma palavra característica** e o classificador está **pouco confiante** entre os 8 temas (abaixo de 0,5).

A primeira verificação foi criada porque palavras comuns, como "melhor", aparecem em algum exemplo de treino e contavam como característica: "qual o melhor jogo de 2026?" recebia a resposta de Wi-Fi. A exigência de 0,4 evita recusar perguntas válidas pouco claras, como "me mandaram um qr code pra pagar". Sem a segunda verificação, "qual o resultado do jogo?" recebia uma resposta de segurança.

Os limiares foram escolhidos olhando só o lote `validacao`, o teste da base e as 18 perguntas fora do escopo de desenvolvimento (seção 5.3), **sem usar o lote `cego`** (seção 5.4).

In [ ]:
PALAVRAS_GENERICAS = set(
    "a o as os um uma uns umas de do da dos das em no na nos nas por para pra pro com sem e ou que se "
    "eu me meu minha meus minhas voce seu sua ele ela isso esse essa este esta isto aqui la "
    "quero queria preciso posso pode fazer faco faz fiz devo como qual quais quando onde porque "
    "recebi recebeu tem ter foi ser estar esta sao ta tao muito mais algo alguem coisa vai vou "
    "ajuda ajudar sobre agora hoje".split()
)


def palavras_relevantes(texto):
    return {p for p in normalizar(texto).split() if p not in PALAVRAS_GENERICAS and len(p) > 2}


CLASSE_FORA = "fora_do_escopo"

# palavras características: aparecem no treino de no máximo 2 temas
# ("jogo", "aplicativo", "celular" aparecem em quase todos e não indicam que o assunto é segurança)
temas_por_palavra = {}
for texto, tema in zip(treino_assistente["pergunta"], treino_assistente["tema"]):
    if tema == CLASSE_FORA:
        continue
    for palavra in palavras_relevantes(texto):
        temas_por_palavra.setdefault(palavra, set()).add(tema)
VOCAB_BASE = {p for p, temas in temas_por_palavra.items() if len(temas) <= 2}
LIMIAR_ESCOPO = 0.5  # sem palavra característica, só responde se o classificador estiver confiante
LIMIAR_FORA = 0.4  # a categoria "fora do escopo" recusa sozinha a partir desta probabilidade


LIMIAR_DUVIDA = 0.35  # abaixo disso o assistente confirma o tema com o usuário (seção 5.4)


def assistente_responder(pergunta):
    if contem_dado_sensivel(pergunta):  # mesma proteção do notebook de exemplo
        return {"tema": "dado_sensivel", "confianca": 1.0, "resposta": RESPOSTA_DADO_SENSIVEL}
    probs = dict(zip(classificador.classes_, classificador.predict_proba([normalizar(pergunta)])[0]))
    p_fora = probs.pop(CLASSE_FORA)
    ordem = sorted(probs, key=probs.get, reverse=True)
    tema = ordem[0]
    confianca = float(probs[tema] / (1 - p_fora))  # probabilidade entre os 8 temas
    caracteristica = bool(palavras_relevantes(pergunta) & VOCAB_BASE)
    if (p_fora > probs[tema] and (p_fora >= LIMIAR_FORA or not caracteristica)) or \
            (not caracteristica and confianca < LIMIAR_ESCOPO):
        return {"tema": "fora_do_escopo", "confianca": confianca, "resposta": RESPOSTA_FORA_DO_ESCOPO}
    r = {"tema": tema, "confianca": confianca, "resposta": RESPOSTAS_ADOLESCENTES[tema]}
    if confianca < LIMIAR_DUVIDA:
        r["duvida"] = [tema, ordem[1]]  # os dois temas mais prováveis
    return r


def formatar_resposta(r):
    # mesmo formato de saída do exemplo: Tema, resposta, Atenção e Referência
    if r["tema"] not in NOMES_TEMAS:
        return r["resposta"]
    aviso = ""
    if "duvida" in r:
        a, b = (NOMES_TEMAS[t] for t in r["duvida"])
        aviso = f"🤔 Não tenho certeza se entendi. Sua pergunta é sobre **{a}** ou **{b}**? Abaixo vai a orientação sobre {a}.\n\n"
    return (f"{aviso}**Tema: {NOMES_TEMAS[r['tema']]}**\n\n{r['resposta']}\n\n"
            f"**Atenção:** {ALERTAS[r['tema']]}\n\n**Referência para consulta:** {REFERENCIAS[r['tema']]}\n\n"
            f"_Material educativo: valide procedimentos atualizados no canal oficial._")


r = assistente_responder("Um amigo me mandou mensagem de outro número pedindo dinheiro, o que faço?")
print("Tema:", r["tema"], f"| confiança: {r['confianca']:.0%}\n")
print(formatar_resposta(r))

**Por que essa resposta? (explicabilidade)** Na regressão logística, cada pedaço de texto tem um peso para cada tema. Somando os pesos dos pedaços de cada palavra, dá para mostrar **quais palavras da pergunta mais pesaram** na escolha do tema. O site mostra essa explicação no botão "Por que essa resposta?".

In [ ]:
uniao = classificador.named_steps["featureunion"]
regressao = classificador.named_steps["logisticregression"]
vet_char, vet_palavra = (vet for _, vet in uniao.transformer_list)
n_char = len(vet_char.vocabulary_)


def explicar(pergunta, n=4):
    # peso de cada palavra = soma dos pesos (para o tema escolhido) dos n-gramas que vêm dela
    texto = normalizar(pergunta)
    tema = assistente_responder(pergunta)["tema"]
    if tema not in NOMES_TEMAS or not texto:
        return []
    c = list(regressao.classes_).index(tema)
    relativo = regressao.coef_[c] - regressao.coef_.mean(axis=0)
    x_char = vet_char.transform([texto]).toarray()[0]
    x_pal = vet_palavra.transform([texto]).toarray()[0]
    analisa_char, analisa_pal = vet_char.build_analyzer(), vet_palavra.build_analyzer()
    total_char = pd.Series(analisa_char(texto)).value_counts()

    pesos = {}
    for palavra in texto.split():
        for ng in analisa_char(palavra):
            j = vet_char.vocabulary_.get(ng)
            if j is not None:
                pesos[palavra] = pesos.get(palavra, 0) + x_char[j] * relativo[j] / total_char[ng]
    for termo in set(analisa_pal(texto)):
        j = vet_palavra.vocabulary_.get(termo)
        if j is not None:
            partes = termo.split()
            for palavra in partes:
                pesos[palavra] = pesos.get(palavra, 0) + x_pal[j] * relativo[n_char + j] / len(partes)
    candidatas = [(p, w) for p, w in pesos.items() if w > 0 and p not in PALAVRAS_GENERICAS and len(p) > 2]
    return [p for p, _ in sorted(candidatas, key=lambda t: -t[1])[:n]]


for p in ["um cara disse q é do suporte do jogo e pediu o codigo q chegou no meu cel",
          "posso entrar no app do banco usando o wifi da escola?"]:
    print(f"{p}\n  → {NOMES_TEMAS[assistente_responder(p)['tema']]} | palavras que mais pesaram: {explicar(p)}")

## 4. Visualizar exemplos e interpretar o score
No **baseline**, o score é a similaridade de cosseno (0 a 1) com a frase de treino mais parecida. No **assistente**, é a probabilidade que o classificador dá ao tema escolhido entre os 8 temas. Nos dois casos o score **não** é certeza de resposta correta nem confiança clínica ou jurídica: é uma medida do modelo, que precisa de calibração com casos reais anonimizados antes de qualquer piloto.

In [ ]:
perguntas = ["Roubaram meu WhatsApp", "Fiz Pix para golpista", "Como faço senha segura?",
             "um cara no zap pediu o código que chegou no meu celular", "Qual o resultado do jogo?", "Senha: 123456"]
for p in perguntas:
    b, a = baseline_responder(p), assistente_responder(p)
    print(f"Pergunta: {p}")
    print(f"  Baseline   → {b['tema']} | score: {b['confianca']:.2f}")
    print(f"  Assistente → {a['tema']} | score: {a['confianca']:.2f}")
    print(f"  Resposta do assistente: {a['resposta'][:150]}...\n")

## 5. Testar o atendimento automaticamente

### 5.1 Conjunto de teste
As 24 perguntas de teste têm situações que os modelos nunca viram. Indicadores do planejamento:

| Indicador | Como medimos |
|---|---|
| Perguntas respondidas corretamente | a resposta entregue é a do tema correto (cair no *fallback* conta como erro) |
| Identificação do tema | acurácia do tema previsto |
| Clareza | seção 5.2 |

In [ ]:
resultados = []
for caso in teste.itertuples():
    b, a = baseline_responder(caso.pergunta), assistente_responder(caso.pergunta)
    resultados.append(dict(caso_id=caso.id_caso, pergunta=caso.pergunta, tema_esperado=caso.tema,
                           baseline=b["tema"], score_baseline=round(b["confianca"], 3), acertou_baseline=b["tema"] == caso.tema,
                           assistente=a["tema"], score_assistente=round(a["confianca"], 3), acertou_assistente=a["tema"] == caso.tema))
resultado_testes = pd.DataFrame(resultados)

resumo = pd.DataFrame({
    "respostas corretas": [f"{resultado_testes[f'acertou_{m}'].sum()}/{len(teste)}" for m in ("baseline", "assistente")],
    "acurácia do tema": [accuracy_score(resultado_testes["tema_esperado"], resultado_testes[m]) for m in ("baseline", "assistente")],
    "caiu no fallback": [f"{(resultado_testes[m] == 'fora_do_escopo').sum()}/{len(teste)}" for m in ("baseline", "assistente")],
}, index=["Baseline (exemplo)", "Assistente adaptado"])
display(resumo)

print("Casos em que algum método falhou:")
display(resultado_testes.loc[~(resultado_testes["acertou_baseline"] & resultado_testes["acertou_assistente"]),
                             ["pergunta", "tema_esperado", "baseline", "assistente"]])

In [ ]:
print("Assistente adaptado: desempenho por tema no teste\n")
print(classification_report(resultado_testes["tema_esperado"], resultado_testes["assistente"], zero_division=0))

### 5.2 Clareza das respostas
Medimos o total de palavras, o tamanho das frases e o índice de legibilidade Flesch adaptado ao português (Martins et al., 1996): quanto maior, mais fácil de ler. Faixas: 75–100 muito fácil, 50–75 fácil, 25–50 difícil, 0–25 muito difícil. É uma estimativa automática (conta sílabas de forma aproximada); a confirmação da H2 vem do teste com adolescentes (seção 9).

In [ ]:
def silabas(palavra):
    # aproximação: cada grupo de vogais conta como uma sílaba
    palavra = unicodedata.normalize("NFKD", palavra.lower())
    palavra = "".join(c for c in palavra if not unicodedata.combining(c))
    return max(1, len(re.findall(r"[aeiouy]+", palavra)))


def metricas_clareza(texto):
    frases = [f for f in re.split(r"[.!?\n]+", texto) if re.search(r"\w", f)]
    palavras = re.findall(r"[A-Za-zÀ-ÿ0-9]+", texto)
    pal_por_frase = len(palavras) / len(frases)
    sil_por_palavra = sum(map(silabas, palavras)) / len(palavras)
    flesch = 248.835 - 1.015 * pal_por_frase - 84.6 * sil_por_palavra
    return {"palavras": len(palavras), "palavras por frase": round(pal_por_frase, 1), "Flesch (pt)": round(flesch, 1)}


clareza = pd.DataFrame({
    "Baseline (exemplo)": pd.DataFrame([metricas_clareza(t) for t in resposta_por_tema]).mean(),
    "Assistente adaptado": pd.DataFrame([metricas_clareza(t) for t in RESPOSTAS_ADOLESCENTES.values()]).mean(),
}).T.round(1)
clareza

As respostas adaptadas são **mais longas** (trazem exemplo e o que fazer se o golpe já aconteceu), mas têm **frases mais curtas** e índice de legibilidade maior, ou seja, são mais fáceis de ler.

### 5.3 Linguagem informal e perguntas fora do escopo
A base de teste usa linguagem formal. Adolescentes escrevem de outro jeito, então criamos algumas perguntas em linguagem informal e outras fora do escopo. Elas **não fazem parte da base oficial**; servem para discutir os limites do projeto.

In [ ]:
perguntas_informais = [
    ("mano um cara no zap disse q é meu primo e pediu pix urgente", "whatsapp"),
    ("chegou msg falando q ganhei skin gratis é só clicar no link", "phishing"),
    ("um cara disse q é do suporte do jogo e pediu o codigo q chegou no meu cel", "falso_suporte"),
    ("acho q hackearam meu insta, oq eu faço", "senha"),
    ("tão mandando um print no grupo da sala dizendo q vai ter greve amanha, é vdd?", "boato"),
    ("posso entrar no app do banco usando o wifi da escola?", "wifi"),
    ("um app de jogo quer acesso aos meus contatos e localização", "privacidade"),
    ("paguei um tenis no insta por pix e o vendedor sumiu", "pix"),
]
# perguntas fora do escopo de desenvolvimento: usadas para escolher os limiares da regra de escopo
perguntas_fora = ["qual a capital da França?", "me ajuda com a lição de matemática", "como faço um bolo de chocolate?",
                  "qual o melhor jogo de 2026?", "qual o melhor aplicativo de música?", "quanto custa um playstation 5?",
                  "qual a música mais tocada da semana?", "como eu passo de fase no minecraft?", "que dia é o feriado de novembro?",
                  "me conta uma piada", "qual o melhor time do brasileirão?", "como faço pra aprender inglês rápido?",
                  "que filme tá passando no cinema?", "como fazer uma redação nota mil?", "quantos anos tem o neymar?",
                  "qual o melhor fone bluetooth barato?", "me ensina a desenhar anime", "como deixar meu cabelo cacheado?"]

casos_extras = perguntas_informais + [(p, "fora_do_escopo") for p in perguntas_fora]
extras = pd.DataFrame([{"pergunta": p, "esperado": esperado,
                        "baseline": baseline_responder(p)["tema"], "assistente": assistente_responder(p)["tema"]}
                       for p, esperado in casos_extras])
print("Acertos do baseline:  ", (extras["baseline"] == extras["esperado"]).sum(), "de", len(extras))
print("Acertos do assistente:", (extras["assistente"] == extras["esperado"]).sum(), "de", len(extras))
extras

O baseline é **muito cauteloso**: recusa bem as perguntas fora do escopo, mas também recusa a maioria das perguntas válidas, porque sem uma palavra-chave exata ou uma frase quase idêntica às de treino ele cai no *fallback*. Para um adolescente, receber "não tenho uma orientação confiável" quando pergunta sobre um link de skin grátis é uma oportunidade perdida de prevenção.

O assistente adaptado também recusa quase todas as perguntas fora do escopo e responde às válidas. A regra de escopo não é perfeita: em "como deixar meu cabelo cacheado?" ele ainda responde com um tema, porque "deixar" aparece em exemplos de privacidade ("deixar o perfil privado").

### 5.4 Validação com perguntas novas
O conjunto de teste da base tem só 8 situações diferentes, escritas em linguagem formal. Para medir o assistente em condições mais próximas do uso real, o grupo escreveu 104 perguntas novas, em linguagem informal (`perguntas_validacao.csv`), em três lotes:
- **validacao** (40 perguntas): escritas antes dos exemplos complementares; revelaram que a primeira versão do assistente acertava só 22 de 40;
- **final** (24 perguntas): escritas antes de treinar a versão atual e usadas **uma única vez**, para medir o resultado sem ajustes. Na versão anterior do assistente (sem os exemplos complementares), o acerto nesse lote era de 14 de 24.
- **cego** (40 perguntas: 4 por tema e 8 fora do escopo): escritas **antes** de criar a categoria "fora do escopo" e sem testar nenhuma delas antes. Com a versão anterior do assistente, o acerto era de 34 de 40: 30 de 32 dentro do escopo, mas só 4 de 8 fora do escopo. Depois das mudanças, o lote foi usado uma única vez: 38 de 40. É a medida mais honesta do projeto, e os 2 erros não foram corrigidos, para não contaminar a medida.

Também comparamos o baseline treinado com os mesmos exemplos complementares, para separar o efeito dos **dados** do efeito do **método**.

In [ ]:
frases_ext = pd.concat([treino["pergunta"], complementares["pergunta"]], ignore_index=True)
temas_ext = pd.concat([treino["tema"], complementares["tema"]], ignore_index=True)
vet_ext = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2)).fit(frases_ext)
mat_ext = vet_ext.transform(frases_ext)

modelos = {
    "Baseline (exemplo)": baseline_responder,
    "Baseline + exemplos complementares": lambda p: baseline_responder(p, vetorizador=vet_ext, matriz=mat_ext,
                                                                       temas_treino=temas_ext),
    "Assistente adaptado": assistente_responder,
}
for nome, funcao in modelos.items():
    validacao[nome] = validacao["pergunta"].map(lambda p: funcao(p)["tema"])

acertos = pd.DataFrame({
    nome: validacao.groupby("lote").apply(lambda g: f"{(g[nome] == g['tema_esperado']).sum()}/{len(g)}")
    for nome in modelos
}).T[["validacao", "final", "cego"]]
display(acertos)

print("Erros do assistente adaptado nas perguntas de validação:")
erros_validacao = validacao[validacao["Assistente adaptado"] != validacao["tema_esperado"]]
display(erros_validacao[["lote", "pergunta", "tema_esperado", "Assistente adaptado"]])

### 5.5 Confirmação do tema quando o assistente está em dúvida
Quando a confiança fica abaixo de **0,35**, o assistente pergunta: *"Sua pergunta é sobre **A** ou **B**?"*, com os dois temas mais prováveis. O limiar foi escolhido com o lote `validacao`, o teste da base e as perguntas informais (72 perguntas), sem usar o lote `final`: nesses casos, **todos os erros tinham o tema certo como segunda opção**, e com 0,35 o assistente pergunta em só 5 das 72 perguntas.

A tabela mostra o acerto considerando que, quando há confirmação, o usuário escolhe o tema certo entre as duas opções.

In [ ]:
def acerto_com_confirmacao(pergunta, esperado):
    r = assistente_responder(pergunta)
    return esperado in r.get("duvida", [r["tema"]])


conjuntos = {
    "Teste da base": list(zip(teste["pergunta"], teste["tema"])),
    "Validação": list(zip(validacao.loc[validacao["lote"] == "validacao", "pergunta"],
                          validacao.loc[validacao["lote"] == "validacao", "tema_esperado"])),
    "Validação final": list(zip(validacao.loc[validacao["lote"] == "final", "pergunta"],
                                validacao.loc[validacao["lote"] == "final", "tema_esperado"])),
    "Teste cego": list(zip(validacao.loc[validacao["lote"] == "cego", "pergunta"],
                           validacao.loc[validacao["lote"] == "cego", "tema_esperado"])),
}
confirmacao = pd.DataFrame({
    nome: {
        "acerto direto": f"{sum(assistente_responder(p)['tema'] == e for p, e in casos)}/{len(casos)}",
        "com confirmação": f"{sum(acerto_com_confirmacao(p, e) for p, e in casos)}/{len(casos)}",
        "vezes que perguntou": f"{sum('duvida' in assistente_responder(p) for p, _ in casos)}/{len(casos)}",
    }
    for nome, casos in conjuntos.items()
}).T
confirmacao

**Como ler:** se o "baseline + exemplos complementares" melhora pouco, a melhoria vem principalmente do **método** (classificador, gírias e regra de escopo); se melhora muito, vem dos **dados**. Nos dois casos, os exemplos complementares fazem parte da melhoria proposta: adaptar o chatbot à forma como adolescentes escrevem.

## 6. Construir o chatbot de demonstração

### 6.1 Demonstração no Gradio (Colab)
A tela apresenta a mensagem do usuário e a resposta. Para reduzir riscos, não salvamos nem exportamos o histórico do chat. Ainda assim, **o ambiente de execução e a plataforma podem registrar dados tecnicamente**: nunca use informações pessoais reais. Evite compartilhar links públicos sem avaliar controles de segurança.

In [ ]:
!pip -q install 'gradio>=4,<6'
import gradio as gr


def conversar(mensagem, historico):
    return formatar_resposta(assistente_responder(mensagem))


with gr.Blocks(title="Fica Esperto AI — Segurança Digital para Adolescentes") as demo:
    gr.Markdown("# 🛡️ Fica Esperto AI\nProtótipo didático, não oficial. Não escreva senha, CPF, códigos ou dados do banco.")
    gr.ChatInterface(fn=conversar, examples=["Um amigo mudou de número e pediu um Pix",
                                             "Recebi um link dizendo que ganhei uma skin grátis",
                                             "Acho que hackearam meu Instagram",
                                             "Posso usar o app do banco no Wi-Fi do shopping?"])
demo.launch(share=False)

### 6.2 Exportar o assistente para o site
Além do Gradio, o projeto tem um site (pasta `site/`) em HTML, CSS e JavaScript, **sem back-end**: o próprio navegador roda o assistente. Esta célula salva em `site/js/modelo.js` tudo o que o site precisa: vocabulário e pesos do classificador, gírias, regras de escopo e de confirmação, respostas, alertas e referências. Também exporta o **baseline**, para o "modo comparação" do chat mostrar as duas respostas lado a lado.

**Sempre que mudar algo no assistente, rode esta célula de novo.** No Colab, baixe o arquivo gerado (painel **Arquivos** → `site/js/modelo.js`) e substitua o da pasta `site/js/`.

In [ ]:
import json

baseline_site = {
    "vocab": {termo: int(i) for termo, i in vetorizador.vocabulary_.items()},
    "idf": [round(float(x), 5) for x in vetorizador.idf_],
    "matriz": [[[int(j), round(float(v), 5)] for j, v in zip(linha.indices, linha.data)] for linha in matriz],
    "temasTreino": list(temas_treino),
    "palavrasTema": PALAVRAS_TEMA,
    "palavrasGolpe": ["golpe", "fraude", "internet", "seguranca", "segurança", "invad"],
    "fallback": FALLBACK,
    "mensagemSensivel": baseline_responder("Senha: 123")["resposta"],
}

modelo_site = {
    "prefixos": PREFIXOS,
    "girias": list(GIRIAS.items()),
    "palavrasGenericas": sorted(PALAVRAS_GENERICAS),
    "vocabBase": sorted(VOCAB_BASE),
    "limiarEscopo": LIMIAR_ESCOPO,
    "limiarFora": LIMIAR_FORA,
    "classeFora": CLASSE_FORA,
    "limiarDuvida": LIMIAR_DUVIDA,
    "baseline": baseline_site,
    "padroesSensiveis": PADROES_DADOS_SENSIVEIS,
    "vetorizadores": [
        {"analisador": vet.analyzer, "ngramas": list(vet.ngram_range),
         "vocab": {termo: int(i) for termo, i in vet.vocabulary_.items()},
         "idf": [round(float(x), 5) for x in vet.idf_]}
        for _, vet in uniao.transformer_list
    ],
    "classes": list(regressao.classes_),
    "coef": [[round(float(x), 5) for x in linha] for linha in regressao.coef_],
    "intercepto": [round(float(x), 5) for x in regressao.intercept_],
    "nomesTemas": NOMES_TEMAS,
    "respostas": RESPOSTAS_ADOLESCENTES,
    "alertas": ALERTAS,
    "referencias": REFERENCIAS,
    "respostasOriginais": resposta_por_tema.to_dict(),  # usadas na página de avaliação
    "respostaForaDoEscopo": RESPOSTA_FORA_DO_ESCOPO,
    "respostaDadoSensivel": RESPOSTA_DADO_SENSIVEL,
}

os.makedirs("site/js", exist_ok=True)
with open("site/js/modelo.js", "w", encoding="utf-8") as f:
    f.write("// Gerado pelo notebook (seção 6.2). Não edite à mão.\n")
    f.write("window.MODELO = " + json.dumps(modelo_site, ensure_ascii=False) + ";\n")

print(f"site/js/modelo.js salvo ({os.path.getsize('site/js/modelo.js') / 1024:.0f} KB)")

## 7. Desenhar o fluxo de atendimento

```text
ADOLESCENTE ABRE O CHAT
      |
      v
Aviso: não escrever senha, CPF, códigos; projeto educativo, não oficial
      |
      v
Pergunta sobre segurança digital (pode ser com gírias e abreviações)
      |
      +--> Contém possível dado sensível? --> Orientar a apagar o dado e reformular
      |
      v
Normalização do texto e das gírias
      |
      +--> Assunto fora dos 8 temas? --> Fallback + adulto de confiança / SaferNet
      |
      v
Classificação do tema
      |
      v
Resposta em passos + exemplo do dia a dia + Atenção + Referência
      |
      v
Adolescente avalia a resposta (👍 / 👎, sem salvar dados)
      |
      v
Encerrar ou procurar adulto de confiança / canal oficial
```

**Urgências:** golpe com dinheiro → avisar um responsável e contatar o banco imediatamente; conta invadida → recuperação oficial do aplicativo; ameaça, chantagem, exposição de fotos ou cyberbullying → adulto de confiança e canal de ajuda da SaferNet Brasil; risco à integridade → serviço de emergência ou autoridade competente. **Nunca** pedir senhas, códigos SMS, documentos ou acesso remoto.

## 8. Plano proposto de implantação junto ao público adolescente
Este cronograma é **apenas uma proposta de projeto**.

In [ ]:
canais = pd.DataFrame([
    ("CH01", "Site do projeto", "Chat aberto no celular ou computador", "Web", "MVP", 1),
    ("CH02", "Aula ou oficina na escola", "Atividade guiada com o professor", "Presencial", "Piloto", 1),
    ("CH03", "Cartazes na escola", "QR code que abre o chat", "Presencial", "Piloto", 1),
    ("CH04", "Plataforma de ensino da escola", "Link no ambiente virtual da turma", "Web", "Futuro", 2),
    ("CH05", "Redes sociais da escola", "Link e dicas semanais", "Social", "Futuro", 2),
    ("CH06", "Aplicativo de mensagens", "Menu de segurança em canal oficial", "Mensageria", "Futuro", 3),
], columns=["canal_id", "canal", "ponto_de_entrada", "tipo", "fase", "complexidade_1a3"])
canais["observacoes"] = "Proposta acadêmica; não representa implantação aprovada."
display(canais[["canal", "ponto_de_entrada", "fase", "complexidade_1a3"]].sort_values(["fase", "complexidade_1a3"]))

### Etapas e critérios para um piloto responsável

**Fase 1 — Preparação:** revisão das respostas por professores e especialistas, autorização da escola e dos responsáveis para atividades com menores de idade, revisão de privacidade (LGPD), acessibilidade, escolha de métricas e definição de encaminhamento para adultos de confiança.

**Fase 2 — MVP controlado:** site disponível em ambiente de teste, sem coleta de dados pessoais, com avaliação anônima das respostas (página `site/avaliacao.html`).

**Fase 3 — Piloto na escola:** oficinas assistidas e cartazes com QR code; testes de compreensão com turmas de idades diferentes e protocolo para situações urgentes (ameaças, exposição, cyberbullying).

**Fase 4 — Decisão de escala:** revisar lacunas de cobertura (novos temas pedidos pelos alunos), medir utilidade e definir quem mantém as respostas atualizadas.

**KPIs recomendados:** cobertura de temas, taxa de *fallback*, respostas avaliadas como úteis, notas de compreensão, falhas de segurança, acessibilidade e mudança observada de comportamento (medida com estudo rigoroso, não presumida).

## 9. Testes de segurança, limitações e próximos passos
- **Não é serviço de emergência nem canal bancário.** Situações críticas devem ser encaminhadas a um adulto de confiança e às instituições oficiais.
- **Proteção de dados:** o reconhecimento por regex detecta apenas alguns formatos (CPF, "senha: ..."); não substitui políticas de privacidade.
- **Cobertura:** o assistente responde apenas aos 8 temas da base de conhecimento.
- **Robustez:** o dicionário de gírias foi montado manualmente e não cobre tudo; a regra de escopo erra para os dois lados. Ainda aceita algumas perguntas fora do tema que usem palavras da base ("como deixar meu cabelo cacheado?") e, desde a categoria "fora do escopo", recusa algumas perguntas válidas mas vagas: no lote `final` passou de 24 para 22 acertos ("como faço pra minha conta ficar mais protegida?" e "no aeroporto tem duas redes com o mesmo nome"). Somando os três lotes, o acerto foi de 96 para 100 de 104. Recusar uma pergunta válida é um erro mais leve do que dar uma orientação errada, porque o assistente pede para explicar de outro jeito.
- **Resultado no teste pode estar otimista:** o vocabulário dos temas e a regra de escopo foram ajustados depois de analisar os erros no conjunto de teste, e os exemplos complementares foram escritos depois de analisar os erros do lote `validacao`. A medida mais honesta é o lote `cego` (seção 5.4), escrito antes das últimas mudanças e usado uma única vez. Mesmo assim, ele foi escrito pelo mesmo grupo que escreveu o treino, e os exemplos fora do escopo cobrem assuntos parecidos (jogos, escola, comida). A confirmação vem do teste com adolescentes.
- **Dados sintéticos:** os exemplos complementares e as perguntas de validação foram escritos pelo grupo, não por adolescentes. Perguntas reais podem ser bem diferentes.
- **Qualidade editorial:** revise cada orientação, os links de referência e a data de revisão com profissionais capacitados.
- **Não substitui** orientação de responsáveis, da escola, do banco ou de profissionais especializados.

### Teste com adolescentes (H1 e H2)
O site tem uma página de avaliação (`site/avaliacao.html`) para aplicar presencialmente. Cada participante:
1. lê **3 situações** e, para cada uma, duas respostas: a resposta revisada original e a adaptada, em ordem aleatória e sem saber qual é qual. Dá notas de 1 a 5 para "entendi o que fazer" e "é fácil de ler" e escolhe a que ajudaria mais → **testa a H2**;
2. escreve **perguntas do jeito dele**, vê a resposta do assistente e diz se ela tinha a ver com a pergunta → **testa a H1 com perguntas novas**.

As respostas ficam salvas só no aparelho usado no teste. No fim, na "Área do aplicador", baixe os dois CSVs e coloque-os na mesma pasta deste notebook.

In [ ]:
if not os.path.exists("avaliacao_comparacoes.csv"):
    print("Ainda não há resultados. Aplique o teste e coloque os CSVs baixados nesta pasta.")
else:
    comp = pd.read_csv("avaliacao_comparacoes.csv")
    livres = pd.read_csv("avaliacao_perguntas.csv") if os.path.exists("avaliacao_perguntas.csv") else pd.DataFrame()
    n = comp["participante"].nunique()
    print(f"Participantes: {n}\n")

    print("H2: notas médias (1 a 5) por versão da resposta")
    display(comp.groupby("versao")[["entendi", "facil"]].agg(["mean", "count"]).round(2))

    preferidas = comp[comp["preferida"] == "sim"]["versao"].value_counts(normalize=True)
    print("\nQual resposta ajudaria mais (% das escolhas):")
    print((preferidas * 100).round(1).astype(str) + "%")

    if len(livres):
        print(f"\nH1: {len(livres)} perguntas escritas pelos participantes")
        print((livres["relevancia"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")
        display(livres[["pergunta", "tema_previsto", "relevancia"]])

    if n < 10:
        print(f"\n⚠️ Só {n} participantes: trate os resultados como indícios, não como conclusão.")

### Conclusões
**H1 (responde corretamente):** o baseline do exemplo erra principalmente por recusar perguntas válidas (*fallback*); o assistente adaptado responde a todas as perguntas do teste e acerta a maioria (seção 5.1).

**H2 (linguagem simples):** as respostas adaptadas têm frases mais curtas e leitura mais fácil (seção 5.2). A confirmação depende do teste com adolescentes.

**Próximos passos:**
- aumentar a base com perguntas escritas pelos próprios adolescentes no teste e treinar de novo, comparando o desempenho antes e depois;
- incluir temas importantes para adolescentes que a base não cobre, como **cyberbullying, sextorsão e golpes em jogos**. Por serem temas sensíveis, as respostas precisam ser escritas e revisadas com especialistas (psicólogos, educadores, SaferNet) antes de entrar no assistente.

**Desafio extra:** comparar diferentes limiares de confirmação (`0.25`, `0.35`, `0.45`) e medir quantas vezes o assistente pergunta sem necessidade. A evolução para um LLM exige curadoria, busca em documentos oficiais (RAG), proteções contra *prompt injection*, avaliação e supervisão.

## 10. Exportar resultados para apresentação
Baixe pelo painel lateral **Arquivos** após rodar a célula.

In [ ]:
resultado_testes.to_csv("resultado_testes.csv", index=False, encoding="utf-8-sig")
indicadores = resumo.join(clareza)
indicadores.to_csv("resumo_indicadores.csv", encoding="utf-8-sig")
print("Salvo: resultado_testes.csv (caso a caso) e resumo_indicadores.csv (tabela para os slides)")
display(indicadores)

### Gráficos para os slides
Cores: azul da SoulCode para o assistente adaptado e vermelho da Claro para o baseline, as mesmas do site. Os gráficos também são salvos em PNG.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# azul SoulCode (assistente) e vermelho Claro (baseline): par validado para daltonismo, as mesmas cores do site
AZUL, LARANJA, TINTA, TINTA_2, GRADE = "#0000ff", "#da291c", "#0b0b0b", "#52514e", "#e1e0d9"

def pct(certos, total):
    return 100 * certos / total

lotes = {
    "Teste da base\n(24 perguntas)": (resultado_testes["acertou_baseline"].sum(), resultado_testes["acertou_assistente"].sum(), len(teste)),
}
for nome, lote in [("Validação\n(40 perguntas)", "validacao"), ("Validação final\n(24 perguntas)", "final"),
                   ("Teste cego\n(40 perguntas)", "cego")]:
    g_ = validacao[validacao["lote"] == lote]
    lotes[nome] = ((g_["Baseline (exemplo)"] == g_["tema_esperado"]).sum(),
                   (g_["Assistente adaptado"] == g_["tema_esperado"]).sum(), len(g_))

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(lotes))
for k, (nome, cor, desloc) in enumerate([("Baseline (exemplo)", LARANJA, 0.17), ("Assistente adaptado", AZUL, -0.17)]):
    valores = [pct(v[k], v[2]) for v in lotes.values()]
    ax.barh(y + desloc, valores, height=0.3, color=cor, label=nome)
    for yi, v in zip(y + desloc, valores):
        ax.text(v + 1.5, yi, f"{v:.0f}%", va="center", fontsize=10, color=TINTA)
ax.set_yticks(y, list(lotes))
ax.invert_yaxis()
ax.set_xlim(0, 110)
ax.set_xlabel("Perguntas respondidas corretamente (%)", color=TINTA_2)
ax.set_title("Baseline × assistente adaptado", loc="left", fontsize=13, color=TINTA)
ax.grid(axis="x", color=GRADE, linewidth=1)
ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=TINTA_2, length=0)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig("grafico_acertos.png", dpi=200)
plt.show()

# matriz de confusão do assistente (teste da base + validação)
temas = sorted(NOMES_TEMAS)
categorias = temas + ["fora_do_escopo"]
esperados = list(resultado_testes["tema_esperado"]) + list(validacao["tema_esperado"])
previstos = list(resultado_testes["assistente"]) + list(validacao["Assistente adaptado"])
matriz_conf = pd.crosstab(pd.Categorical(esperados, categorias), pd.Categorical(previstos, categorias), dropna=False)
azuis = LinearSegmentedColormap.from_list("azuis", ["#fcfcfb", "#ccccff", "#8080ff", "#0000ff", "#000080"])

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(matriz_conf.values, cmap=azuis)
rotulos = [NOMES_TEMAS[t] for t in temas]
ax.set_xticks(range(matriz_conf.shape[1]), rotulos + ["Fora do escopo"], rotation=40, ha="right")
ax.set_yticks(range(len(categorias)), rotulos + ["Fora do escopo"])
limite = matriz_conf.values.max() / 2
for i in range(matriz_conf.shape[0]):
    for j in range(matriz_conf.shape[1]):
        v = matriz_conf.values[i, j]
        if v:
            ax.text(j, i, v, ha="center", va="center", color="#ffffff" if v > limite else TINTA, fontsize=10)
ax.set_xlabel("Tema escolhido pelo assistente", color=TINTA_2)
ax.set_ylabel("Tema correto", color=TINTA_2)
ax.set_title("Onde o assistente acerta e onde confunde (teste + validação)", loc="left", fontsize=12, color=TINTA)
ax.tick_params(length=0, colors=TINTA_2)
for lado in ax.spines.values():
    lado.set_visible(False)
plt.tight_layout()
plt.savefig("grafico_confusao.png", dpi=200)
plt.show()
print("Salvo: grafico_acertos.png e grafico_confusao.png")

### Números para a página "Sobre" do site
A página `site/sobre.html` lê os resultados de `site/js/metricas.js`. Rode esta célula sempre que o assistente mudar, para o site mostrar os números atualizados.

In [ ]:
def contagem(certos, total):
    return {"certos": int(certos), "total": int(total)}

metricas_site = {
    "conjuntos": [
        {"nome": nome.replace("\n", " "), "baseline": contagem(v[0], v[2]), "assistente": contagem(v[1], v[2])}
        for nome, v in lotes.items()
    ],
    "fallbackTeste": {"baseline": int((resultado_testes["baseline"] == "fora_do_escopo").sum()),
                      "assistente": int((resultado_testes["assistente"] == "fora_do_escopo").sum()),
                      "total": len(teste)},
    "clareza": {modelo: {k: float(v) for k, v in linha.items()} for modelo, linha in clareza.iterrows()},
    "informais": {"baseline": int((extras["baseline"] == extras["esperado"]).sum()),
                  "assistente": int((extras["assistente"] == extras["esperado"]).sum()), "total": len(extras)},
    "treino": {"base": len(treino), "complementares": len(exemplos), "temas": len(NOMES_TEMAS)},
}
with open("site/js/metricas.js", "w", encoding="utf-8") as f:
    f.write("// Gerado pelo notebook (seção 10). Não edite à mão.\n")
    f.write("window.METRICAS = " + json.dumps(metricas_site, ensure_ascii=False, indent=1) + ";\n")
print("site/js/metricas.js salvo")